# Loading Hospital Data from CSV Files


In [0]:
# Read the Hospital General Information CSV
hospital_info = spark.read.csv(
    "/Volumes/workspace/default/hospital_assessment/Hospital_General_Information.csv",
    header=True,
    inferSchema=True
)

# Create a temporary view for SQL queries
hospital_info.createOrReplaceTempView("hospital_info")

print(f"Loaded {hospital_info.count()} rows")
display(hospital_info.limit(5))

In [0]:
# Read the Complications and Deaths CSV
complications = spark.read.csv(
    "/Volumes/workspace/default/hospital_assessment/Complications_and_Deaths-Hospital.csv",
    header=True,
    inferSchema=True
)

complications.createOrReplaceTempView("complications")
print(f"Loaded {complications.count()} rows")

In [0]:
# Read the HCAHPS Patient Survey CSV
patient_survey = spark.read.csv(
    "/Volumes/workspace/default/hospital_assessment/HCAHPS-Hospital_Patient_Survey.csv",
    header=True,
    inferSchema=True
)

patient_survey.createOrReplaceTempView("patient_survey")
print(f"Loaded {patient_survey.count()} rows")

In [0]:
# Read the Healthcare Associated Infections CSV
infections = spark.read.csv(
    "/Volumes/workspace/default/hospital_assessment/Healthcare_Associated_Infections-Hospital.csv",
    header=True,
    inferSchema=True
)

infections.createOrReplaceTempView("infections")
print(f"Loaded {infections.count()} rows")

In [0]:
# Read the Medicare Hospital Spending CSV
medicare_spending = spark.read.csv(
    "/Volumes/workspace/default/hospital_assessment/Medicare_Hospital_Spending_by_Claim.csv",
    header=True,
    inferSchema=True
)

medicare_spending.createOrReplaceTempView("medicare_spending")
print(f"Loaded {medicare_spending.count()} rows")

In [0]:
# Read the Timely and Effective Care CSV
timely_care = spark.read.csv(
    "/Volumes/workspace/default/hospital_assessment/Timely_and_Effective_Care-Hospital.csv",
    header=True,
    inferSchema=True
)

timely_care.createOrReplaceTempView("timely_care")
print(f"Loaded {timely_care.count()} rows")

In [0]:
# Read the Unplanned Hospital Visits CSV
unplanned_visits = spark.read.csv(
    "/Volumes/workspace/default/hospital_assessment/Unplanned_Hospital_Visits-Hospital.csv",
    header=True,
    inferSchema=True
)

unplanned_visits.createOrReplaceTempView("unplanned_visits")
print(f"Loaded {unplanned_visits.count()} rows")

## Available SQL Tables

* `hospital_info` - Hospital General Information
* `complications` - Complications and Deaths
* `patient_survey` - HCAHPS Patient Survey results
* `infections` - Healthcare Associated Infections
* `medicare_spending` - Medicare Hospital Spending by Claim
* `timely_care` - Timely and Effective Care measures
* `unplanned_visits` - Unplanned Hospital Visits

In [0]:
%sql
select * from complications limit 5


In [0]:
%sql
-- Layer 0: Raw normalization of the data

CREATE OR REPLACE TEMP VIEW raw_complications AS
SELECT 'Facility ID' AS Facility_ID
, 'Measure ID' AS Measure_ID
, 'Score' AS Score
FROM complications
;

CREATE OR REPLACE TEMP VIEW raw_readmissions AS
SELECT 'Facility ID' AS Facility_ID
, 'Measure ID' AS Measure_ID
, 'Score' AS Score
FROM unplanned_visits
;

CREATE OR REPLACE TEMP VIEW raw_infections AS
SELECT 'Facility ID' AS Facility_ID
, 'Measure ID' AS Measure_ID
, 'Score' AS Score
FROM infections
;

CREATE OR REPLACE TEMP VIEW raw_te AS
SELECT 'Facility ID' AS Facility_ID
, 'Measure ID' AS Measure_ID
, 'Score' AS Score
FROM infections
;

CREATE OR REPLACE TEMP VIEW raw_cost AS
SELECT 'Facility ID' AS Facility_ID
, 'Period' AS Period
, 'Avg Spndg Per EP Hospital' AS Avg_Spnd_Per_EP
FROM medicare_spending
;

CREATE OR REPLACE TEMP VIEW raw_survey AS
SELECT 'Facility ID' AS Facility_ID
, 'HCAHPS Measure ID' AS Measure_ID
, 'Patient Survey Star Rating' AS Score
FROM patient_survey
;

CREATE OR REPLACE TEMP VIEW raw_hospital AS
SELECT 'Facility ID' AS Facility_ID
, 'Facility Name' AS Facility_Name
, 'City/Town' AS City_Town
, 'State' AS State
, 'ZIP Code' AS ZIP_Code
, 'Hospital Type' AS Hospital_Type
, 'Hospital Ownership' AS Hospital_Ownership
, 'Emergency Services' AS Emergency_Services
, 'Hospital overall rating' AS Hospital_Overall_Rating
, 'Count of Facility MORT Measures' AS Cnt_Mort
, 'Count of Facility Safety Measures' AS Cnt_Safety
, 'Count of Facility READM Measures' AS Cnt_Readm
, 'Count of Facility Pt Exp Measures' AS Cnt_Ptexp
, 'Count of Facility TE Measures' AS Cnt_Te
FROM hospital_info
;

In [0]:
%sql
-- Layer 1: Putting measures together in one long table, z-scoring by domain, and calculating overall z-score
-- Note: CMS uses exactly the same measures to calculate outcomes scores for its star ratings.

-- one long table
CREATE OR REPLACE TEMP VIEW measures_long AS
SELECT Facility_ID
, Measure_ID
, CASE WHEN Measure_ID IN ('MORT_30_AMI','MORT_30_CABG', 
                    'MORT_30_COPD','MORT_30_HF',
                    'MORT_30_PN','MORT_30_STK','PSI_04')
        THEN 'mort'
        WHEN Measure_ID IN ('EDAC_30_AMI','EDAC_30_HF',
                     'EDAC_30_PN','READM_30_CABG',
                     'READM_30_COPD','READM_30_HIP_KNEE',
                     'Hybrid_HWR','OP_32','OP_35_ADM',
                     'OP_35_ED','OP_36')
        THEN 'readm'
        WHEN Measure_ID IN ('HAI_1_SIR','HAI_2_SIR',
                      'HAI_3_SIR','HAI_4_SIR',
                      'HAI_5_SIR','HAI_6_SIR', 
                      'COMP_HIP_KNEE', 'PSI_90')
        THEN 'safety'
    END AS Domain
, try_cast(Score AS double) AS Score
FROM  (
    SELECT Facility_ID
    , Measure_ID
    , Score
    FROM raw_complications
    UNION ALL
    SELECT Facility_ID
    , Measure_ID
    , Score
    FROM raw_readmissions
    UNION ALL
    SELECT Facility_ID
    , Measure_ID
    , Score
    FROM raw_infections
) AS U
WHERE Measure_ID IN ('MORT_30_AMI','MORT_30_CABG', 
                    'MORT_30_COPD','MORT_30_HF',
                    'MORT_30_PN','MORT_30_STK','PSI_04',
                    'EDAC_30_AMI','EDAC_30_HF',
                    'EDAC_30_PN','READM_30_CABG',
                    'READM_30_COPD','READM_30_HIP_KNEE',
                    'Hybrid_HWR','OP_32','OP_35_ADM',
                    'OP_35_ED','OP_36',
                    'HAI_1_SIR','HAI_2_SIR',
                    'HAI_3_SIR','HAI_4_SIR',
                    'HAI_5_SIR','HAI_6_SIR', 
                    'COMP_HIP_KNEE', 'PSI_90')
;

-- domain z-scores
CREATE OR REPLACE TEMP VIEW measure_z AS
SELECT Facility_ID
, Domain
, Measure_ID
, -1.0 * (Score - avg(Score) OVER (PARTITION BY Measure_ID)) / stddev_samp(Score) OVER (PARTITION BY Measure_ID) AS Z
FROM measures_long
;

-- average domain scores then overall z-score
CREATE OR REPLACE TEMP VIEW domain_master_z AS
WITH dmean AS (
    SELECT Facility_ID
    , Domain
    , avg(Z) AS Z_Mean
    FROM measure_z
    GROUP BY Facility_ID, Domain
    )
SELECT Facility_ID
, Domain
, (Z_Mean - avg(Z_Mean) OVER (PARTITION BY Domain)) / 
    stddev_samp(Z_Mean) OVER (PARTITION BY Domain) AS Master_Z
FROM dmean
;


In [0]:
%sql
-- Layer 2: Using the hospital data to calculate thresholds
-- Note: For the CMS star rating methodology, a hospital is included if it has 3 categories with at least 3 ratings per category, provided either Complications or Infections (or both) are represented. For this analysis, a hospital is included if it has at least 2 of the 3 categories (Complications, Infections, and Readmissions) with at least 3 ratings in the category.

CREATE OR REPLACE TEMP VIEW hospital_clean AS
WITH flagged AS (
    SELECT Facility_ID
    , Facility_Name
    , City_Town
    , State
    , lpad(CAST (ZIP_Code AS string),5,'0') AS ZIP_Code
    , try_cast(Hospital_Overall_Rating AS DOUBLE) AS Hospital_Overall_Rating
    , Hospital_Type
    , Hospital_Ownership
    , Emergency_Services
    , CASE WHEN try_cast(Cnt_Mort AS DOUBLE) >= 3
            THEN 1
            ELSE 0
            END AS Mort_Flag
    , CASE WHEN try_cast(Cnt_Readm AS DOUBLE) >= 3
            THEN 1
            ELSE 0
            END AS Readm_Flag
    , CASE WHEN try_cast(Cnt_Safety AS DOUBLE) >= 3
            THEN 1
            ELSE 0
            END AS Safety_Flag
    , CASE WHEN try_cast(Cnt_Ptexp AS DOUBLE) >= 3
            THEN 1
            ELSE 0
            END AS Ptexp_Flag
    , CASE WHEN try_cast(Cnt_Te AS DOUBLE) >= 3
            THEN 1
            ELSE 0
            END AS Te_Flag
    FROM raw_hospital
    )
SELECT *
, (Mort_Flag + Readm_Flag + Safety_Flag) AS Include_Score
, CASE (Ptexp_Flag + Te_Flag + Mort_Flag + Readm_Flag + Safety_Flag)
    WHEN 3
    THEN 'Small'
    WHEN 4
    THEN 'Medium'
    WHEN 5 THEN 'Large'
    END AS Size_Indicator
FROM flagged
WHERE (Mort_Flag + Readm_Flag + Safety_Flag) >= 2
;

In [0]:
%sql
-- Layer 3: Creating the Outcomes score by removing facilities with insufficient numbers of scores and averaging the remaining domain scores (Complications, Readmissions, and Infections) per facility

CREATE OR REPLACE TEMP VIEW outcomes AS
SELECT d.Facility_ID
, AVG(CASE d.Domain WHEN 'mort'
                    THEN CASE WHEN h.Mort_Flag = 1 
                            THEN d.Master_Z
                            END
                    WHEN 'readm'
                    THEN CASE WHEN h.Readm_Flag = 1
                            THEN d.Master_Z
                            END
                    WHEN 'safety'
                    THEN CASE WHEN h.Safety_Flag = 1
                            THEN d.Master_Z
                            END
                    END) AS Outcomes
FROM domain_master_z d
JOIN hospital_clean h
    ON d.Facility_ID = h.Facility_ID
GROUP BY d.Facility_ID
;

In [0]:
%sql
-- Layer 4: Preparing Cost table for joining by padding Facility_ID and creating Per_Episode_Cost column

CREATE OR REPLACE TEMP VIEW cost AS
SELECT lpad(cast(Facility_ID AS STRING), 6, '0') AS Facility_ID
, SUM(CASE WHEN Period = 'Complete Episode'
        THEN try_cast(Avg_Spnd_Per_Ep AS DOUBLE)
        ELSE 0
        END) AS Per_Episode_Cost
FROM raw_cost
GROUP BY lpad(cast(Facility_ID AS STRING), 6, '0')
;

In [0]:
%sql
-- Layer 5: Calculating Census Division from State and calculating Value Quadrant using median Outcomes and median Per_Episode_Cost
-- Note: facilities without cost information will be dropped because both Outcomes and Per_Episode_Cost are required for the Value Quadrant creation.

-- Creating Census Division 
CREATE OR REPLACE TEMP VIEW census_div (State, Census_Division) AS VALUES
('CT','New England'), ('ME','New England'), ('MA','New England'), ('NH','New England'), ('RI','New England'), ('VT','New England'),('NJ','Middle Atlantic'),('NY','Middle Atlantic'), ('PA','Middle Atlantic'),('IL','East North Central'),('IN','East North Central'),('MI','East North Central') ,('OH','East North Central'), ('WI','East North Central'), ('IA','West North Central'), ('KS','West North Central'), ('MN','West North Central'), ('MO','West North Central'), ('ND','West North Central'), ('SD','West North Central'), ('NE','West North Central'), ('DE','South Atlantic'),('DC','South Atlantic' ),('FL','South Atlantic'), ('GA','South Atlantic'),('MD','South Atlantic'),('NC','South Atlantic'),('SC','South Atlantic'),('VA', 'South Atlantic'),('WV','South Atlantic'), ('AL','East South Central'), ('KY','East South Central'), ('MS','East South Central'), ('TN','East South Central'), ('AR','West South Central'), ('LA','West South Central'), ('OK','West South Central'), ('TX','West South Central'), ('AZ','Mountain'), ('CO','Mountain'), ('ID','Mountain'), ('MT','Mountain'), ('NV','Mountain'),('NM','Mountain'), ('UT','Mountain'), ('WY','Mountain'), ('CA','Pacific'), ('HI','Pacific'),  ('OR','Pacific'), ('WA','Pacific')
;

CREATE OR REPLACE TEMP VIEW hospital_value AS
WITH base AS (
    SELECT h.Facility_ID
    , h.Facility_Name
    , h.City_Town
    , h.State
    , cd.Census_Division
)

